In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:
# ==================== REVISED DATASET PREPARATION ====================
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset, Dataset
import pandas as pd
import os

# 1. Login to HF
hf_token = userdata.get('HF_TOKEN')
if hf_token:
    login(token=hf_token)
    print("✅ Successfully logged in to Hugging Face!")
else:
    print("❌ HF_TOKEN not found in secrets.")

# 2. Load Train and Test splits directly from Hugging Face
print("⏳ Loading datasets from Hugging Face...")
try:
    train_ds = load_dataset("alibayram/doktorsitesi", split="train")
    test_ds = load_dataset("alibayram/doktorsitesi", split="test")
    train_df = pd.DataFrame(train_ds)
    test_df = pd.DataFrame(test_ds)
    print("✅ Train and Test splits loaded successfully!")
except Exception as e:
    print(f"⚠️ Error loading splits: {e}")
    print("Note: If the repository only contains a 'train' split, this will fail.")
    raise e

# 3. Filter required medical fields
required_fields = [
    "beyin-ve-sinir-cerrahisi", "uroloji", "ortopedi-ve-travmatoloji",
    "genel-cerrahi", "kulak-burun-bogaz-hastaliklari",
    "fiziksel-tip-ve-rehabilitasyon", "kardiyoloji", "kalp-damar-cerrahisi"
]

def filter_specialities(df):
    dahiliye_mask = df["doctor_speciality"].str.startswith("dahiliye-ve-ic-hastaliklari", na=False)
    return df[dahiliye_mask | df["doctor_speciality"].isin(required_fields)].reset_index(drop=True)

train_filtered = filter_specialities(train_df)
test_filtered = filter_specialities(test_df)

print(f"Filtered Train size: {len(train_filtered)} | Filtered Test size: {len(test_filtered)}")

# 4. Define target sizes
TARGET_TRAIN_SIZE = 20000
TARGET_TEST_SIZE = 2000

# Sample train set
train_final = train_filtered.sample(
    n=min(TARGET_TRAIN_SIZE, len(train_filtered)),
    random_state=28
)

# 5. Stratify Test set to exactly match Train set distribution
# Get the exact percentage of each class in the training set
train_distribution = train_final["doctor_speciality"].value_counts(normalize=True)

test_samples = []
for speciality, proportion in train_distribution.items():
    # Calculate how many samples we need for this class in the test set
    n_needed = int(round(proportion * TARGET_TEST_SIZE))

    # Get available rows in the test set for this class
    available_in_test = test_filtered[test_filtered["doctor_speciality"] == speciality]

    # Sample safely (don't try to take more than what exists in the test split)
    n_to_take = min(n_needed, len(available_in_test))
    if n_to_take > 0:
        test_samples.append(available_in_test.sample(n=n_to_take, random_state=42))

# Combine and shuffle the final test set
test_final = pd.concat(test_samples).sample(frac=1, random_state=28).reset_index(drop=True)

print(f"\n🎯 Final Train shape: {train_final.shape}")
print(f"🎯 Final Test shape: {test_final.shape}")

# Verify distributions match visually (Showing COUNTS and percentages for all classes)
print("\n📊 Distribution Check (All Classes):")
dist_df = pd.DataFrame({
    "Train Count": train_final["doctor_speciality"].value_counts(),
    "Train %": train_final["doctor_speciality"].value_counts(normalize=True) * 100,
    "Test Count": test_final["doctor_speciality"].value_counts(),
    "Test %": test_final["doctor_speciality"].value_counts(normalize=True) * 100
}).fillna(0) # Fills 0 if a class is entirely missing from test

# Convert counts to integer so they display cleanly (e.g., 250 instead of 250.0)
dist_df["Train Count"] = dist_df["Train Count"].astype(int)
dist_df["Test Count"] = dist_df["Test Count"].astype(int)

# Use .to_string() to force pandas to print every row without truncating
print(dist_df.round(2).to_string())

# 6. Save to Disk so downstream cells can use it
data_save_path = "/content/drive/MyDrive/CSE4078_Project/dataset_splits_hf"
os.makedirs(data_save_path, exist_ok=True)

Dataset.from_pandas(train_final).save_to_disk(f"{data_save_path}/train_dataset")
Dataset.from_pandas(test_final).save_to_disk(f"{data_save_path}/test_dataset")

print(f"\n💾 Datasets successfully aligned and saved to {data_save_path}!")

✅ Successfully logged in to Hugging Face!
⏳ Loading datasets from Hugging Face...
✅ Train and Test splits loaded successfully!
Filtered Train size: 57758 | Filtered Test size: 14483

🎯 Final Train shape: (20000, 4)
🎯 Final Test shape: (1999, 4)

📊 Distribution Check (All Classes):
                                                                       Train Count  Train %  Test Count  Test %
doctor_speciality                                                                                              
beyin-ve-sinir-cerrahisi                                                      3846    19.23         385   19.26
uroloji                                                                       3114    15.57         311   15.56
ortopedi-ve-travmatoloji                                                      2212    11.06         221   11.06
genel-cerrahi                                                                 2054    10.27         205   10.26
dahiliye-ve-ic-hastaliklari-nefroloji         

Saving the dataset (0/1 shards):   0%|          | 0/20000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1999 [00:00<?, ? examples/s]


💾 Datasets successfully aligned and saved to /content/drive/MyDrive/CSE4078_Project/dataset_splits_hf!
